---
## Pandas

NumPy is great for homogeneous numeric data, but real datasets have **labeled columns of mixed types** (names, dates, prices, categories) and **missing values**. That's where **Pandas** comes in.

The two core data structures:

| Structure | Description | Analogy |
|---|---|---|
| **`Series`** | 1-D labeled array (values + index) | One column of a spreadsheet |
| **`DataFrame`** | 2-D table of Series sharing an index | The whole spreadsheet / SQL table |

### Series — a labeled 1-D array

In [1]:
import pandas as pd
import numpy as np
print("Pandas version:", pd.__version__)

Pandas version: 3.0.3


In [4]:
# Default index: 0, 1, 2, ...
s = pd.Series([10, 20, 30, 40])
print(s)

0    10
1    20
2    30
3    40
dtype: int64


In [5]:
# Custom index — labels become first-class citizens
pop = pd.Series([1444, 1393, 332, 273],
                index=["India", "China", "USA", "Indonesia"],
                name="population_millions")
print(pop)

print("\nLabel access  pop['India'] :", pop["India"])
print("Position access pop.iloc[0]:", pop.iloc[0])
print("\nSlicing by label (INCLUSIVE of endpoint!):")
print(pop["China":"USA"])

India        1444
China        1393
USA           332
Indonesia     273
Name: population_millions, dtype: int64

Label access  pop['India'] : 1444
Position access pop.iloc[0]: 1444

Slicing by label (INCLUSIVE of endpoint!):
China    1393
USA       332
Name: population_millions, dtype: int64


In [6]:
# A Series behaves like a NumPy array (vectorized ops, boolean masks)...
print(pop * 1_000_000)              # convert to absolute numbers
print("\nover 500M:\n", pop[pop > 500])

# ...and like a dict
print("\n'USA' in pop:", "USA" in pop)
print(pd.Series({"a": 1, "b": 2, "c": 3}))   # build straight from a dict

India        1444000000
China        1393000000
USA           332000000
Indonesia     273000000
Name: population_millions, dtype: int64

over 500M:
 India    1444
China    1393
Name: population_millions, dtype: int64

'USA' in pop: True
a    1
b    2
c    3
dtype: int64


> 🔑 **Key idea — index alignment.** When you combine two Series, Pandas aligns them **by label**, not by position. Labels present in only one of them produce `NaN`.

In [2]:
q1 = pd.Series({"laptops": 120, "phones": 340, "tablets": 90})
q2 = pd.Series({"phones": 410, "laptops": 150, "watches": 60})   # different order + keys

print(q1 + q2)    # aligned by label; 'tablets' & 'watches' → NaN

laptops    270.0
phones     750.0
tablets      NaN
watches      NaN
dtype: float64


###  DataFrame — the workhorse

A **DataFrame** is a dict-like collection of Series that share the same row index. Common ways to build one:

In [3]:
# From a dict of lists (most common for small examples)
df = pd.DataFrame({
    "name":   ["Asha", "Bilal", "Chen", "Diya", "Evan", "Farah"],
    "dept":   ["IT", "HR", "IT", "Finance", "HR", "IT"],
    "age":    [25, 32, 28, 41, 36, 29],
    "salary": [52000, 48000, 61000, 75000, 50000, 58000],
    "city":   ["Agra", "Delhi", "Mumbai", "Delhi", "Agra", "Mumbai"],
})
df

,name,dept,age,salary,city
0,Asha,IT,25,52000,Agra
1,Bilal,HR,32,48000,Delhi
2,Chen,IT,28,61000,Mumbai
3,Diya,Finance,41,75000,Delhi
4,Evan,HR,36,50000,Agra
5,Farah,IT,29,58000,Mumbai


In [4]:
# First look at any DataFrame — your standard inspection toolkit
print("shape  :", df.shape)
print("columns:", list(df.columns))
print("index  :", df.index)

df.head(3)        # first 3 rows (df.tail() → last rows)

shape  : (6, 5)
columns: ['name', 'dept', 'age', 'salary', 'city']
index  : RangeIndex(start=0, stop=6, step=1)


,name,dept,age,salary,city
0,Asha,IT,25,52000,Agra
1,Bilal,HR,32,48000,Delhi
2,Chen,IT,28,61000,Mumbai


In [12]:
df.info()         # dtypes, non-null counts, memory — ALWAYS run this on new data

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   name    6 non-null      str  
 1   dept    6 non-null      str  
 2   age     6 non-null      int64
 3   salary  6 non-null      int64
 4   city    6 non-null      str  
dtypes: int64(2), str(3)
memory usage: 372.0 bytes


In [13]:
df.describe()     # summary statistics for numeric columns

,age,salary
count,6.000000,6.000000
mean,31.833333,57333.333333
std,5.845226,9953.223933
min,25.000000,48000.000000
25%,28.250000,50500.000000
50%,30.500000,55000.000000
75%,35.000000,60250.000000
max,41.000000,75000.000000


### 2.3 Pandas Data Types

Each **column** has one dtype. Pandas dtypes extend NumPy's:

| dtype | Used for |
|---|---|
| `int64`, `float64` | numbers |
| `bool` | True/False |
| `object` / `str` | text (or mixed) |
| `datetime64[ns]` | timestamps |
| `category` | repeated labels (saves memory, enables ordering) |

Wrong dtypes are one of the **most common sources of bugs**: numbers read as text can't be summed, dates read as strings can't be compared. Always check `df.dtypes` and convert with `astype()`, `pd.to_numeric()`, `pd.to_datetime()`.

In [7]:
raw = pd.DataFrame({
    "order_id": ["1001", "1002", "1003"],
    "amount":   ["2500", "1800", "3200"],        # numbers stored as strings!
    "date":     ["2026-01-05", "2026-01-09", "2026-02-14"],
    "city":     ["Agra", "Delhi", "Agra"],
})
print(raw.dtypes)
raw["amount"].sum() #not sums

order_id    str
amount      str
date        str
city        str
dtype: object


'250018003200'

In [15]:
# Fix the dtypes
clean = raw.copy()
clean["amount"] = pd.to_numeric(clean["amount"])
clean["date"]   = pd.to_datetime(clean["date"])
clean["city"]   = clean["city"].astype("category")

print(clean.dtypes)
print("\nreal sum:", clean["amount"].sum())
print("orders in January:", (clean["date"].dt.month == 1).sum())

order_id               str
amount               int64
date        datetime64[us]
city              category
dtype: object

real sum: 7500
orders in January: 2


### 2.4 Selecting Data: columns, `loc`, `iloc`

| Syntax | Selects | By |
|---|---|---|
| `df["col"]` | one column → Series | label |
| `df[["c1","c2"]]` | multiple columns → DataFrame | label |
| `df.loc[rows, cols]` | rows & cols | **labels** (slices *inclusive*) |
| `df.iloc[rows, cols]` | rows & cols | **integer positions** (slices *exclusive*) |
| `df[mask]` / `df.loc[mask]` | rows where mask is True | boolean |

> Rule of thumb: **`loc` = label, `iloc` = integer**. Avoid plain chained indexing like `df[...][...] = value` when assigning.

In [8]:
# From a dict of lists (most common for small examples)
df = pd.DataFrame({
    "name":   ["Asha", "Bilal", "Chen", "Diya", "Evan", "Farah"],
    "dept":   ["IT", "HR", "IT", "Finance", "HR", "IT"],
    "age":    [25, 32, 28, 41, 36, 29],
    "salary": [52000, 48000, 61000, 75000, 50000, 58000],
    "city":   ["Agra", "Delhi", "Mumbai", "Delhi", "Agra", "Mumbai"],
})
df

,name,dept,age,salary,city
0,Asha,IT,25,52000,Agra
1,Bilal,HR,32,48000,Delhi
2,Chen,IT,28,61000,Mumbai
3,Diya,Finance,41,75000,Delhi
4,Evan,HR,36,50000,Agra
5,Farah,IT,29,58000,Mumbai


In [17]:
people = df.set_index("name")      # use names as row labels for nicer demos
people

,dept,age,salary,city
name,,,,
Asha,IT,25,52000,Agra
Bilal,HR,32,48000,Delhi
Chen,IT,28,61000,Mumbai
Diya,Finance,41,75000,Delhi
Evan,HR,36,50000,Agra
Farah,IT,29,58000,Mumbai


In [18]:
print("--- loc: by LABEL ---")
print(people.loc["Chen"])                          # one row → Series
print()
print(people.loc[["Asha", "Diya"], ["dept", "salary"]])
print()
print(people.loc["Bilal":"Diya", "age":"salary"])  # label slices are INCLUSIVE

--- loc: by LABEL ---
dept          IT
age           28
salary     61000
city      Mumbai
Name: Chen, dtype: object

         dept  salary
name                 
Asha       IT   52000
Diya  Finance   75000

       age  salary
name              
Bilal   32   48000
Chen    28   61000
Diya    41   75000


In [19]:
print("--- iloc: by POSITION ---")
print(people.iloc[0])                  # first row
print()
print(people.iloc[0:3, [0, 2]])        # rows 0,1,2  × columns 0 and 2 (exclusive stop)
print()
print(people.iloc[-1, -1])             # bottom-right cell

--- iloc: by POSITION ---
dept         IT
age          25
salary    52000
city       Agra
Name: Asha, dtype: object

      dept  salary
name              
Asha    IT   52000
Bilal   HR   48000
Chen    IT   61000

Mumbai


In [20]:
# Boolean filtering — exactly like NumPy masks
print(people[people["salary"] > 55000])
print()
print(people[(people["dept"] == "IT") & (people["age"] < 30)])
print()
# query() is a readable alternative for complex filters
print(people.query("dept == 'IT' and age < 30"))

          dept  age  salary    city
name                               
Chen        IT   28   61000  Mumbai
Diya   Finance   41   75000   Delhi
Farah       IT   29   58000  Mumbai

      dept  age  salary    city
name                           
Asha    IT   25   52000    Agra
Chen    IT   28   61000  Mumbai
Farah   IT   29   58000  Mumbai

      dept  age  salary    city
name                           
Asha    IT   25   52000    Agra
Chen    IT   28   61000  Mumbai
Farah   IT   29   58000  Mumbai


In [23]:
# Adding / transforming columns (vectorized — no loops!)
people = people.copy()
people["salary_lakh"] = people["salary"] / 100_000
people["senior"] = people["age"] >= 35
people["band"] = np.where(people["salary"] >= 55000, "High", "Standard")
people

,dept,age,salary,city,salary_lakh,senior,band
name,,,,,,,
Asha,IT,25,52000,Agra,0.52,False,Standard
Bilal,HR,32,48000,Delhi,0.48,False,Standard
Chen,IT,28,61000,Mumbai,0.61,False,High
Diya,Finance,41,75000,Delhi,0.75,True,High
Evan,HR,36,50000,Agra,0.50,True,Standard
Farah,IT,29,58000,Mumbai,0.58,False,High


### 2.5 Missing Data (`NaN`)

Real data is messy. Pandas marks missing values as `NaN` (or `NaT` for dates). Core tools:

- `isna()` / `notna()` — detect
- `dropna()` — remove rows/columns with missing values
- `fillna(value)` — replace (constant, mean, forward-fill `ffill()`, ...)

In [24]:
messy = pd.DataFrame({
    "product": ["Pen", "Book", "Bag", "Lamp", "Mug"],
    "price":   [20, np.nan, 850, 1200, np.nan],
    "rating":  [4.1, 4.5, np.nan, 4.8, 3.9],
})
print(messy)
print("\nmissing per column:\n", messy.isna().sum())

  product   price  rating
0     Pen    20.0     4.1
1    Book     NaN     4.5
2     Bag   850.0     NaN
3    Lamp  1200.0     4.8
4     Mug     NaN     3.9

missing per column:
 product    0
price      2
rating     1
dtype: int64


In [25]:
print("dropna() — lose any row with a NaN:\n", messy.dropna())
print()

filled = messy.copy()
filled["price"]  = filled["price"].fillna(filled["price"].mean())   # impute mean
filled["rating"] = filled["rating"].fillna(filled["rating"].median())
print("after filling:\n", filled)

dropna() — lose any row with a NaN:
   product   price  rating
0     Pen    20.0     4.1
3    Lamp  1200.0     4.8

after filling:
   product   price  rating
0     Pen    20.0     4.1
1    Book   690.0     4.5
2     Bag   850.0     4.3
3    Lamp  1200.0     4.8
4     Mug   690.0     3.9


> 💡 **Note:** dropping is safest when missing rows are few and random; filling (imputation) preserves data but injects assumptions. Always count missing values *before* deciding.

### ✏️ Quick Pandas Exercises

1. From `df`, select the `name` and `city` of everyone in HR using `loc` + a boolean mask.
2. Add a column `tax` equal to 10% of salary for salaries above 55,000 and 5% otherwise (`np.where`).
3. In `messy`, fill missing `price` with 0 but **drop** rows missing `rating`.

---
## Data Wrangling: Combining & Grouping

Real analysis almost never happens on a single clean table. You'll constantly:

- **concatenate** tables (stack datasets collected separately),
- **merge/join** tables (combine related information, like SQL joins),
- **group & aggregate** (split → apply → combine), and
- **pivot** (reshape for reporting).

### Concatenation — stacking tables

`pd.concat([...])` glues DataFrames along an axis. `axis=0` stacks rows (default), `axis=1` puts tables side by side aligned on the index.

In [27]:
jan = pd.DataFrame({"order_id": [1, 2, 3],
                    "amount":   [250, 400, 150]})
feb = pd.DataFrame({"order_id": [4, 5],
                    "amount":   [500, 320]})

all_orders = pd.concat([jan, feb], ignore_index=True)   # reset 0..n-1 index
all_orders

,order_id,amount
0,1,250
1,2,400
2,3,150
3,4,500
4,5,320


### 3.2 Merging & Joining — `pd.merge`

`pd.merge(left, right, on="key", how=...)` works like SQL joins:

| `how=` | Keeps |
|---|---|
| `"inner"` *(default)* | only keys present in **both** tables |
| `"left"` | **all** rows of the left table (+ matches from right) |
| `"right"` | all rows of the right table |
| `"outer"` | union of all keys; non-matches get `NaN` |

We'll use a tiny e-commerce schema: a **customers** table and an **orders** table linked by `cust_id`.

In [29]:
customers = pd.DataFrame({
    "cust_id": [101, 102, 103, 104],
    "name":    ["Asha", "Bilal", "Chen", "Diya"],
    "city":    ["Agra", "Delhi", "Mumbai", "Delhi"],
})

orders = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5],
    "cust_id":  [101, 103, 101, 105, 102],   # 105 has no customer record!
    "amount":   [2500, 1800, 3200, 700, 1500],
})

print(customers, "\n")
print(orders)

   cust_id   name    city
0      101   Asha    Agra
1      102  Bilal   Delhi
2      103   Chen  Mumbai
3      104   Diya   Delhi 

   order_id  cust_id  amount
0         1      101    2500
1         2      103    1800
2         3      101    3200
3         4      105     700
4         5      102    1500


In [30]:
# INNER join: only customers who actually placed orders, only orders with a known customer
inner = pd.merge(orders, customers, on="cust_id", how="inner")
inner

,order_id,cust_id,amount,name,city
0,1,101,2500,Asha,Agra
1,2,103,1800,Chen,Mumbai
2,3,101,3200,Asha,Agra
3,5,102,1500,Bilal,Delhi


In [31]:
# LEFT join: keep ALL orders; unknown customers (105) show NaN
left = pd.merge(orders, customers, on="cust_id", how="left")
left

,order_id,cust_id,amount,name,city
0,1,101,2500,Asha,Agra
1,2,103,1800,Chen,Mumbai
2,3,101,3200,Asha,Agra
3,4,105,700,NaN,NaN
4,5,102,1500,Bilal,Delhi


In [32]:
# OUTER join: everything from both sides — great for spotting mismatches
outer = pd.merge(orders, customers, on="cust_id", how="outer", indicator=True)
outer

,order_id,cust_id,amount,name,city,_merge
0,1.0,101,2500.0,Asha,Agra,both
1,3.0,101,3200.0,Asha,Agra,both
2,5.0,102,1500.0,Bilal,Delhi,both
3,2.0,103,1800.0,Chen,Mumbai,both
4,NaN,104,NaN,Diya,Delhi,right_only
5,4.0,105,700.0,NaN,NaN,left_only


The `indicator=True` column `_merge` tells you where each row came from — invaluable for **data-quality audits**: `right_only` = customer who never ordered (Diya), `left_only` = order with a missing/invalid customer id (105).

In [33]:
# Different key names? use left_on / right_on.  Joining on the index? use join().
emp  = pd.DataFrame({"emp_id": [1, 2, 3], "name": ["P", "Q", "R"], "dept_code": ["D1", "D2", "D1"]})
dept = pd.DataFrame({"code": ["D1", "D2"], "dept_name": ["Sales", "Tech"]})

pd.merge(emp, dept, left_on="dept_code", right_on="code", how="left").drop(columns="code")

,emp_id,name,dept_code,dept_name
0,1,P,D1,Sales
1,2,Q,D2,Tech
2,3,R,D1,Sales


> ⚠️ **Duplicate keys multiply rows.** If a key appears *m* times on the left and *n* times on the right, the merge produces *m × n* rows for that key. Always sanity-check `len(result)` after merging; `validate="one_to_many"` etc. can enforce expectations.

### 3.3 GroupBy — Split → Apply → Combine ⭐

`groupby` is the single most powerful Pandas tool. The mental model:

1. **Split** the rows into groups by some key(s),
2. **Apply** a function to each group (sum, mean, count, custom...),
3. **Combine** the results into a new table.

In [34]:
sales = pd.DataFrame({
    "region":  ["North", "South", "North", "East", "South", "East", "North", "South"],
    "product": ["Pen", "Pen", "Book", "Book", "Bag", "Pen", "Bag", "Book"],
    "units":   [120, 80, 45, 60, 25, 95, 30, 50],
    "revenue": [2400, 1600, 6750, 9000, 21250, 1900, 25500, 7500],
})
sales

,region,product,units,revenue
0,North,Pen,120,2400
1,South,Pen,80,1600
2,North,Book,45,6750
3,East,Book,60,9000
4,South,Bag,25,21250
5,East,Pen,95,1900
6,North,Bag,30,25500
7,South,Book,50,7500


In [35]:
# One key, one aggregation
print(sales.groupby("region")["revenue"].sum())
print()
# Several aggregations at once
sales.groupby("region")["revenue"].agg(["count", "sum", "mean", "max"])

region
East     10900
North    34650
South    30350
Name: revenue, dtype: int64



,count,sum,mean,max
region,,,,
East,2,10900,5450.000000,9000
North,3,34650,11550.000000,25500
South,3,30350,10116.666667,21250


In [36]:
# Multiple keys → hierarchical (MultiIndex) result
g = sales.groupby(["region", "product"])["revenue"].sum()
print(g)
print()
print(g.loc["North"])          # drill into one outer group

region  product
East    Book        9000
        Pen         1900
North   Bag        25500
        Book        6750
        Pen         2400
South   Bag        21250
        Book        7500
        Pen         1600
Name: revenue, dtype: int64

product
Bag     25500
Book     6750
Pen      2400
Name: revenue, dtype: int64


In [37]:
# Named aggregations: different functions on different columns, clean output names
summary = (sales
           .groupby("region")
           .agg(total_revenue=("revenue", "sum"),
                avg_units=("units", "mean"),
                n_sales=("product", "count"))
           .sort_values("total_revenue", ascending=False))
summary

,total_revenue,avg_units,n_sales
region,,,
North,34650,65.000000,3
South,30350,51.666667,3
East,10900,77.500000,2


In [38]:
# transform() returns a result ALIGNED to the original rows —
# perfect for "compare each row to its group"
sales = sales.copy()
sales["region_avg"] = sales.groupby("region")["revenue"].transform("mean")
sales["pct_of_region"] = (sales["revenue"] / sales.groupby("region")["revenue"]
                          .transform("sum") * 100).round(1)
sales

,region,product,units,revenue,region_avg,pct_of_region
0,North,Pen,120,2400,11550.000000,6.9
1,South,Pen,80,1600,10116.666667,5.3
2,North,Book,45,6750,11550.000000,19.5
3,East,Book,60,9000,5450.000000,82.6
4,South,Bag,25,21250,10116.666667,70.0
5,East,Pen,95,1900,5450.000000,17.4
6,North,Bag,30,25500,11550.000000,73.6
7,South,Book,50,7500,10116.666667,24.7


In [39]:
# filter() keeps/drops WHOLE groups by a condition
big_regions = sales.groupby("region").filter(lambda g: g["revenue"].sum() > 20000)
print("regions with > ₹20,000 total revenue:")
print(big_regions[["region", "product", "revenue"]])

regions with > ₹20,000 total revenue:
  region product  revenue
0  North     Pen     2400
1  South     Pen     1600
2  North    Book     6750
4  South     Bag    21250
6  North     Bag    25500
7  South    Book     7500


**The three groupby verbs, side by side:**

| Method | Output shape | Typical use |
|---|---|---|
| `.agg()` | one row **per group** | summaries / reports |
| `.transform()` | same shape **as input** | new columns: group means, ranks, % of group |
| `.filter()` | subset of original rows | keep only groups satisfying a condition |

### 3.4 Pivot Tables & Crosstabs

A pivot table is a groupby reshaped into a 2-D grid — one key on rows, another on columns.

In [9]:
sales = pd.DataFrame({
    "region":  ["North", "South", "North", "East", "South", "East", "North", "South"],
    "product": ["Pen", "Pen", "Book", "Book", "Bag", "Pen", "Bag", "Book"],
    "units":   [120, 80, 45, 60, 25, 95, 30, 50],
    "revenue": [2400, 1600, 6750, 9000, 21250, 1900, 25500, 7500],
})
sales

,region,product,units,revenue
0,North,Pen,120,2400
1,South,Pen,80,1600
2,North,Book,45,6750
3,East,Book,60,9000
4,South,Bag,25,21250
5,East,Pen,95,1900
6,North,Bag,30,25500
7,South,Book,50,7500


In [40]:
pivot = sales.pivot_table(index="region", columns="product",
                          values="revenue", aggfunc="sum",
                          fill_value=0, margins=True, margins_name="TOTAL")
pivot

product,Bag,Book,Pen,TOTAL
region,,,,
East,0,9000,1900,10900
North,25500,6750,2400,34650
South,21250,7500,1600,30350
TOTAL,46750,23250,5900,75900


In [41]:
# pd.crosstab — frequency counts between two categorical columns
pd.crosstab(sales["region"], sales["product"])

product,Bag,Book,Pen
region,,,
East,0,1,1
North,1,1,1
South,1,1,1
